## Analytical - code verification against 1-D steady state advection diffusion reaction 

This notebook verifies the implementation of `brinkmanPisoFoam` for the following scenario:
- Rectangular channel
- No porous media
- Only a first order, dissolution reaction $A \rightarrow B$ with a known first order constant reaction rate $k$. The reaction rate corresponds to the A parameter in the constant/reactions dictionary. All the other parameters are zero for a constant reaction rate.

The concentration profile is sampled at the line $lc(z): (0, 0, z)$. Owing to the small dimensions of the channel, the Reynolds number is small and the flow is laminar. Therefore, the velocity profile is roughly parabolic (see CFD). Nevertheless, after the entrance length the velocity of the centerline is constant until the end of the domain.

The tests compares the concentration profile in the centerline at steady-state of the CFD simulation, $c_{CFD}$, against the analytical solution for the 1-D advection diffusion reaction equation,

$$ c = c_0\exp{\left(1- \sqrt{1 + 4 k D_{AB} / v_0^2}  \frac{v_0 z}{2D_{AB}} \right)}$$

The characteristic velocity is set to the centerline velocity at the end exit of the channel, $v_0 = v_z(0, 0, z=L)$, where $L$ is the channel length.

The source data must be generated as a `plotOverLine` in paraview at the end of the simulation. For this test, the simulation time is 30 s. An alternative version of the verification test can be performed with the ADR_verification.py script. The script sources the centerline data from a function object, calculates MSE and MAPE and plots the profile of the component A at the steady-state.

In [ ]:
# Scientific computing
import numpy as np

# Facilitate CFD data handling
import pandas as pd

# Visualisation
import matplotlib.pyplot as plt

Define analytical solution

In [ ]:
# Maximum concentration, to be calculated
c0 = 0.01

# Reaction constant
k = 2.5

# Diffusivity, to be updated
D = 1.569e-5


# The domain is 14mm long
L = 0.014

# Number of vertical elements
n_z = 100

# Vertical coordinate vector to sample solution
z = np.linspace(0, L, n_z)

# Source CFD data
df = pd.read_csv('data_vs_z_SS.csv')
z_CFD = df['arc_length']
c_CFD = df['A']
# Centerline velocity v(0,0,z) at the steadystate / m / s
v = df['U:2'].values[-1]

def c_an(z, c0, k, D, v):
    ''' Concentration profile in steady state for 1-D
    advection diffusion reaction equation'''

    return c0 * np.exp( - (np.sqrt(1 + (4*k*D/v**2)) - 1) * (v*z/(2*D)) )

# Construction of analytical profiles
z_1D = z
c_1D = c_an(z_1D, c0, k , D, v)

#### Parameter fitting

In [ ]:
from scipy.optimize import curve_fit

# Define the function for curve_fit to use
def fit_func(z, k):
    return c_an(z, c0, k, D, v)

# Initial guess for k
k_initial = 0.1

# Use curve_fit to fit k
popt, pcov = curve_fit(fit_func, z_CFD, c_CFD, p0=k_initial)

# The optimized value of k is the first element of the returned optimal parameters
k_optimized = popt[0]

print(f"Optimized reaction rate k_opt = %.3f " % k_optimized)
print(f"Reaction rate directly input to OpenFOAM, k = %.3f" % k)

# Error calculation and printing
MSE = np.sqrt(sum((c_an(z_CFD, c0, k, D ,v) - c_CFD)**2)/len(c_CFD))
MAPE = np.sqrt(sum( ((c_an(z_CFD, c0, k, D ,v) - c_CFD)/c_CFD)**2)/len(c_CFD))
print("The mean squared error of molar fraction of A is MSE = %.3e" % MSE)
print("The mean average percentage error of molar fraction of A is MAPE = %.3f %%" % MAPE)

In [ ]:
# Visualisation
plt.plot(z_1D,c_an(z, c0, k, D ,v), label='1-D,SS, k=%.3f' % k)
plt.plot(z_1D,c_an(z, c0, k_optimized, D ,v), label="1-D,SS, k=%.3f" % k_optimized)
plt.plot(z_CFD, c_CFD, label='CFD')

plt.title("1-D ADR Verification: MSE = %.2e, MAPE = %.2f%%" % (MSE, MAPE))

plt.xlabel('z')
plt.ylabel(r'$w_{NaOH}$')
plt.legend()
plt.savefig("verification.jpg")